## 🎯 Learning Objectives
* Design a robust and production-ready prompt for a complex task.
* Apply advanced prompt engineering techniques such as few-shot learning, Chain-of-Thought (CoT), and structured output specification.
* Develop strategies for handling edge cases and ensuring output consistency.
* Evaluate prompt performance based on accuracy, robustness, and adherence to format.


# LLM03-L10: Exercise: Design a Robust Prompt for a Production Task

## Task: Customer Support Ticket Classification and Entity Extraction

In a production environment, Large Language Models (LLMs) are increasingly used to automate initial processing of customer support tickets. This exercise challenges you to design a robust prompt for an LLM to perform the following tasks on incoming customer support tickets:

1.  **Classify** the ticket into one of several predefined categories.
2.  **Extract** key entities relevant to the issue.
3.  **Assign** a priority level.
4.  **Summarize** the core issue concisely.

Your goal is to create a prompt that is highly reliable, consistent, and handles various types of inputs, including ambiguous or irrelevant tickets.

### Scenario Details:

Imagine you are building an AI agent for a SaaS company that provides project management software. Customers submit tickets for various issues.

### Requirements:

Your prompt must achieve the following:

1.  **Input**: Accept a raw customer support ticket text.
2.  **Output Format**: The LLM's response *must* be a valid JSON object with the following structure:
    ```json
    {
      "category": "<string>",
      "sub_category": "<string>" | null,
      "priority": "<string>",
      "entities": [
        {
          "type": "<string>",
          "value": "<string>"
        }
      ],
      "summary": "<string>",
      "reasoning": "<string>" // Explain the classification and priority decision
    }
    ```
3.  **Predefined Categories**: The `category` field must be one of: "Technical Support", "Billing Inquiry", "Account Management", "Feature Request", "General Inquiry", "Spam/Irrelevant".
4.  **Predefined Priorities**: The `priority` field must be one of: "High", "Medium", "Low".
5.  **Entity Types**: Examples of `entity.type` could be "User ID", "Project Name", "Order ID", "Feature Name", "Error Code", "Product Name". You can define more as needed.
6.  **Robustness**: The prompt should be resilient to variations in ticket phrasing, typos, and incomplete information.
7.  **Edge Cases**: Explicitly handle:
    *   **Ambiguous tickets**: If the category is unclear, classify as "General Inquiry" and explain why in `reasoning`.
    *   **Irrelevant/Spam tickets**: Classify as "Spam/Irrelevant" and assign "Low" priority.
8.  **Techniques**: Utilize at least two advanced prompt engineering techniques (e.g., few-shot examples, Chain-of-Thought, role-playing, explicit constraints).
9.  **Clarity**: The prompt should be easy to understand for the LLM.

### Evaluation Criteria:

Your solution will be evaluated based on:

*   **Accuracy**: Correct classification, priority, and entity extraction for diverse tickets.
*   **Consistency**: Adherence to the specified JSON output format across all test cases.
*   **Robustness**: How well it handles edge cases and variations.
*   **Prompt Efficiency**: Conciseness and clarity of the prompt without sacrificing performance.
*   **Reasoning Quality**: The `reasoning` field should provide insightful justification for the LLM's decisions.

Good luck!


In [ ]:
import json
from typing import List, Dict, Any, Optional

# --- Mock LLM Client (for demonstration purposes) ---
# In a real scenario, this would interface with an actual LLM API (e.g., OpenAI, Anthropic, Google Gemini).
class MockLLMClient:
    def __init__(self, model_name: str = "mock-llm-2026-01"): # Assuming a modern LLM model
        self.model_name = model_name
        print(f"Initialized MockLLMClient with model: {self.model_name}")

    def generate(self, prompt: str, temperature: float = 0.7, max_tokens: int = 1024) -> str:
        """
        Simulates an LLM API call. For this exercise, it will return a predefined
        response for specific inputs or a generic structured response.
        In a real application, this would send the prompt to an actual LLM.
        """
        print("\n--- Mock LLM Call Initiated ---")
        print(f"Prompt length: {len(prompt)} characters")
        # print(f"Prompt snippet: {prompt[:200]}...") # Uncomment for debugging prompt

        # Simple heuristic to return a plausible JSON for demonstration
        # In a real scenario, the LLM would generate this based on the prompt.
        if "I can't log in to my account" in prompt:
            return json.dumps({
                "category": "Technical Support",
                "sub_category": "Login Issue",
                "priority": "High",
                "entities": [
                    {"type": "Issue", "value": "login"},
                    {"type": "System", "value": "account"}
                ],
                "summary": "User unable to log in to their account.",
                "reasoning": "The ticket explicitly states a login issue, which is a critical technical support problem."
            }, indent=2)
        elif "My credit card was charged twice" in prompt:
            return json.dumps({
                "category": "Billing Inquiry",
                "sub_category": "Duplicate Charge",
                "priority": "High",
                "entities": [
                    {"type": "Payment Method", "value": "credit card"},
                    {"type": "Issue", "value": "charged twice"}
                ],
                "summary": "Customer reports a duplicate charge on their credit card.",
                "reasoning": "This is a direct billing issue involving a duplicate charge, requiring immediate attention."
            }, indent=2)
        elif "Can you add a dark mode feature?" in prompt:
            return json.dumps({
                "category": "Feature Request",
                "sub_category": null,
                "priority": "Low",
                "entities": [
                    {"type": "Feature Name", "value": "dark mode"}
                ],
                "summary": "User requests a new dark mode feature.",
                "reasoning": "The ticket clearly asks for a new feature, which is a common request and typically lower priority."
            }, indent=2)
        elif "Hello, I am a Nigerian prince" in prompt:
            return json.dumps({
                "category": "Spam/Irrelevant",
                "sub_category": null,
                "priority": "Low",
                "entities": [],
                "summary": "Irrelevant message identified as spam.",
                "reasoning": "The content matches known spam patterns and is not related to product support."
            }, indent=2)
        elif "I need help with something, but I'm not sure what." in prompt:
            return json.dumps({
                "category": "General Inquiry",
                "sub_category": null,
                "priority": "Medium",
                "entities": [],
                "summary": "User needs help but provides insufficient details for specific classification.",
                "reasoning": "The ticket is too vague to assign a specific category, defaulting to general inquiry."
            }, indent=2)
        else:
            # Generic fallback for other inputs, assuming the prompt guides the LLM
            # to produce this structure.
            return json.dumps({
                "category": "General Inquiry",
                "sub_category": null,
                "priority": "Medium",
                "entities": [],
                "summary": "Could not precisely classify the issue based on the provided text. Further investigation may be needed.",
                "reasoning": "The mock client returned a default response as no specific match was found for the input ticket."
            }, indent=2)


# Initialize the mock client
llm_client = MockLLMClient()

# --- Sample Customer Support Tickets ---
sample_tickets = [
    "I can't log in to my account. I keep getting an 'authentication failed' error after the last update. My user ID is 'john.doe'.",
    "My credit card was charged twice for order #XYZ789. This is unacceptable! Please fix it immediately.",
    "Can you add a dark mode feature to the project dashboard? It would be much easier on the eyes at night.",
    "Hello, I am a Nigerian prince and I need your help to transfer a large sum of money. Please reply to my email.",
    "The 'export to CSV' button isn't working on Project 'Alpha'. I tried refreshing, but no luck. This is urgent for my report due tomorrow.",
    "I need help with something, but I'm not sure what. It's about the new feature you released last week.",
    "How do I change my notification settings? I'm getting too many emails.",
    "I forgot my password for user 'jane.smith'. Can you reset it for me?"
]

# --- Expected Output Structure (for reference) ---
# This is what your prompt should guide the LLM to produce.
expected_output_schema = {
    "type": "object",
    "properties": {
        "category": {"type": "string", "enum": ["Technical Support", "Billing Inquiry", "Account Management", "Feature Request", "General Inquiry", "Spam/Irrelevant"]},
        "sub_category": {"type": ["string", "null"]},
        "priority": {"type": "string", "enum": ["High", "Medium", "Low"]},
        "entities": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "type": {"type": "string"},
                    "value": {"type": "string"}
                },
                "required": ["type", "value"]
            }
        },
        "summary": {"type": "string"},
        "reasoning": {"type": "string"}
    },
    "required": ["category", "priority", "entities", "summary", "reasoning"]
}

print("\nSetup complete. You can now proceed to design your prompt.")


## Your Turn: Implement `classify_ticket`

Now, it's your turn to design the prompt and implement the `classify_ticket` function. Your function should take a raw ticket text and use the `llm_client` to get a structured JSON response.

Think carefully about:
*   **Role-playing**: How can you instruct the LLM to act as an expert?
*   **Instructions**: What are the clearest, most unambiguous instructions you can give?
*   **Constraints**: How do you enforce the categories, priorities, and JSON format?
*   **Few-shot examples**: Can you provide 1-2 examples to guide the LLM's output?
*   **Chain-of-Thought**: Should the LLM think step-by-step before providing the final answer?
*   **Error Handling**: How do you instruct the LLM to handle ambiguous or irrelevant inputs?

Iterate on your prompt. Test it with the `sample_tickets` provided in the setup cell. Aim for high accuracy and consistent JSON output.


In [ ]:
import json
from typing import List, Dict, Any, Optional

# Re-initialize the mock client if this cell is run independently
# In a real notebook, it would be defined in the setup cell.
class MockLLMClient:
    def __init__(self, model_name: str = "mock-llm-2026-01"):
        self.model_name = model_name

    def generate(self, prompt: str, temperature: float = 0.7, max_tokens: int = 1024) -> str:
        # This mock client is designed to simulate a good LLM response
        # based on the prompt structure. It uses simple keyword matching
        # to provide more relevant mock outputs than a generic fallback.
        # In a real scenario, the actual LLM would process the prompt.

        prompt_lower = prompt.lower()

        if "i can't log in" in prompt_lower or "authentication failed" in prompt_lower or "forgot my password" in prompt_lower:
            user_id_match = next((word for word in prompt.split() if 'user id' in word.lower() or 'user' in word.lower() and '.' in word), None)
            if not user_id_match:
                user_id_match = next((word.strip("'.") for word in prompt.split() if '@' in word and '.' in word), None)
            user_id_entity = {"type": "User ID", "value": user_id_match.strip("'") if user_id_match else "unknown"}
            return json.dumps({
                "category": "Technical Support",
                "sub_category": "Login/Account Access",
                "priority": "High",
                "entities": [user_id_entity, {"type": "Issue", "value": "login issue"}],
                "summary": "User is experiencing issues logging into their account.",
                "reasoning": "The ticket clearly indicates a critical login or account access problem, requiring immediate technical assistance."
            }, indent=2)
        elif "charged twice" in prompt_lower or "billing" in prompt_lower or "invoice" in prompt_lower:
            order_id_match = next((word for word in prompt.split() if 'order #' in word.lower()), None)
            order_id_entity = {"type": "Order ID", "value": order_id_match.replace('order #', '')} if order_id_match else None
            entities = []
            if order_id_entity: entities.append(order_id_entity)
            entities.append({"type": "Issue", "value": "duplicate charge"})
            return json.dumps({
                "category": "Billing Inquiry",
                "sub_category": "Duplicate Charge",
                "priority": "High",
                "entities": entities,
                "summary": "Customer reports a duplicate charge on their account.",
                "reasoning": "This is a critical billing issue involving incorrect charges, which needs urgent resolution."
            }, indent=2)
        elif "feature request" in prompt_lower or "can you add" in prompt_lower or "would be great if" in prompt_lower:
            feature_name_match = next((word for word in prompt_lower.split() if 'feature' in word or 'mode' in word), None)
            feature_entity = {"type": "Feature Name", "value": feature_name_match} if feature_name_match else None
            entities = []
            if feature_entity: entities.append(feature_entity)
            return json.dumps({
                "category": "Feature Request",
                "sub_category": None,
                "priority": "Low",
                "entities": entities,
                "summary": "User is requesting a new feature for the product.",
                "reasoning": "The ticket clearly expresses a desire for a new product capability, classifying it as a feature request."
            }, indent=2)
        elif "nigerian prince" in prompt_lower or "transfer money" in prompt_lower or "spam" in prompt_lower:
            return json.dumps({
                "category": "Spam/Irrelevant",
                "sub_category": None,
                "priority": "Low",
                "entities": [],
                "summary": "Irrelevant message identified as spam or phishing attempt.",
                "reasoning": "The content matches known spam patterns and is entirely unrelated to product support."
            }, indent=2)
        elif "export to csv" in prompt_lower or "button isn't working" in prompt_lower or "error" in prompt_lower:
            project_name_match = next((word.strip("'") for word in prompt.split() if 'project' in word.lower()), None)
            project_entity = {"type": "Project Name", "value": project_name_match} if project_name_match else None
            entities = []
            if project_entity: entities.append(project_entity)
            entities.append({"type": "Issue", "value": "functionality not working"})
            return json.dumps({
                "category": "Technical Support",
                "sub_category": "Functionality Issue",
                "priority": "High" if "urgent" in prompt_lower else "Medium",
                "entities": entities,
                "summary": "User reports a specific feature (export to CSV) is not working in a project.",
                "reasoning": "This is a clear technical issue affecting core functionality, with urgency indicated."
            }, indent=2)
        elif "notification settings" in prompt_lower or "too many emails" in prompt_lower:
            return json.dumps({
                "category": "Account Management",
                "sub_category": "Notification Settings",
                "priority": "Medium",
                "entities": [
                    {"type": "Setting", "value": "notification settings"}
                ],
                "summary": "User needs assistance with managing their notification preferences.",
                "reasoning": "The ticket is about personal account settings, specifically notifications."
            }, indent=2)
        else:
            # Default for ambiguous or general inquiries
            return json.dumps({
                "category": "General Inquiry",
                "sub_category": None,
                "priority": "Medium",
                "entities": [],
                "summary": "The ticket is too vague or general to be classified into a specific category. Further human review may be required.",
                "reasoning": "The content of the ticket does not clearly fit into any specific predefined category, so it's classified as a general inquiry."
            }, indent=2)

llm_client = MockLLMClient()

sample_tickets = [
    "I can't log in to my account. I keep getting an 'authentication failed' error after the last update. My user ID is 'john.doe'.",
    "My credit card was charged twice for order #XYZ789. This is unacceptable! Please fix it immediately.",
    "Can you add a dark mode feature to the project dashboard? It would be much easier on the eyes at night.",
    "Hello, I am a Nigerian prince and I need your help to transfer a large sum of money. Please reply to my email.",
    "The 'export to CSV' button isn't working on Project 'Alpha'. I tried refreshing, but no luck. This is urgent for my report due tomorrow.",
    "I need help with something, but I'm not sure what. It's about the new feature you released last week.",
    "How do I change my notification settings? I'm getting too many emails.",
    "I forgot my password for user 'jane.smith'. Can you reset it for me?"
]


def classify_ticket(ticket_text: str, client: MockLLMClient) -> Dict[str, Any]:
    """
    Designs a robust prompt for an LLM to classify a customer support ticket,
    extract entities, assign priority, and summarize the issue.
    """

    # --- Prompt Design --- 
    # This prompt incorporates several advanced techniques:
    # 1. Role-playing: "You are an expert Customer Support AI Agent..."
    # 2. Clear Instructions: Explicitly states the task and desired output.
    # 3. Structured Output: Defines the exact JSON schema required.
    # 4. Constraints: Lists allowed categories and priorities.
    # 5. Few-shot Examples: Provides two clear examples to guide the LLM's understanding.
    # 6. Chain-of-Thought (CoT): "Think step-by-step..." encourages logical processing.
    # 7. Edge Case Handling: Instructions for ambiguous/irrelevant tickets.

    prompt = f"""
    You are an expert Customer Support AI Agent for a SaaS company providing project management software. Your task is to analyze incoming customer support tickets and provide a structured JSON response. 

    Follow these instructions precisely:

    1.  **Analyze the User's Request**: Understand the core issue, intent, and urgency.
    2.  **Categorize**: Assign the ticket to one of the following categories:
        -   "Technical Support": For software bugs, errors, or functionality issues.
        -   "Billing Inquiry": For questions or issues related to payments, invoices, or subscriptions.
        -   "Account Management": For user profile changes, password resets, or notification settings.
        -   "Feature Request": For suggestions for new features or improvements.
        -   "General Inquiry": For questions that don't fit other categories or are too vague.
        -   "Spam/Irrelevant": For unsolicited messages, marketing, or content unrelated to product support.
    3.  **Assign Priority**: Determine the urgency and assign one of:
        -   "High": Critical issues affecting core functionality, data loss, or immediate business impact.
        -   "Medium": Important issues, minor bugs, or questions requiring timely resolution.
        -   "Low": General inquiries, feature requests, or non-urgent matters.
    4.  **Extract Entities**: Identify key pieces of information like User IDs, Project Names, Order IDs, Feature Names, Error Codes, etc. If no relevant entities are found, return an empty list.
    5.  **Summarize**: Provide a concise, one-sentence summary of the main issue.
    6.  **Reasoning**: Explain your classification and priority decisions step-by-step.
    7.  **Output Format**: Your final output MUST be a valid JSON object. Do NOT include any other text, comments, or explanations outside the JSON block. The JSON structure must be:
        ```json
        {{
          "category": "<string>",
          "sub_category": "<string>" | null, // More specific category if applicable, otherwise null
          "priority": "<string>",
          "entities": [
            {{
              "type": "<string>", // e.g., "User ID", "Project Name", "Order ID", "Feature Name", "Error Code"
              "value": "<string>"
            }}
          ],
          "summary": "<string>",
          "reasoning": "<string>" // Step-by-step justification for decisions
        }}
        ```
    8.  **Edge Cases**: 
        -   If the ticket is ambiguous, classify as "General Inquiry" and explain the ambiguity in `reasoning`.
        -   If the ticket is clearly spam or irrelevant, classify as "Spam/Irrelevant" and assign "Low" priority.

    --- Examples (Few-Shot Learning) ---

    **Example 1 Ticket:**
    "I can't log in to my account. I keep getting an 'authentication failed' error after the last update. My user ID is 'alice.wonder'."

    **Example 1 Response:**
    ```json
    {{
      "category": "Technical Support",
      "sub_category": "Login Issue",
      "priority": "High",
      "entities": [
        {{"type": "Issue", "value": "authentication failed"}},
        {{"type": "User ID", "value": "alice.wonder"}}
      ],
      "summary": "User is unable to log in due to an authentication failed error.",
      "reasoning": "The ticket explicitly states a login issue with an error message and user ID, indicating a critical technical support problem. The 'authentication failed' error directly points to a login issue, making it high priority."
    }}
    ```

    **Example 2 Ticket:**
    "My credit card was charged twice for order #ABC123. This is unacceptable! Please fix it immediately."

    **Example 2 Response:**
    ```json
    {{
      "category": "Billing Inquiry",
      "sub_category": "Duplicate Charge",
      "priority": "High",
      "entities": [
        {{"type": "Payment Method", "value": "credit card"}},
        {{"type": "Order ID", "value": "ABC123"}}
      ],
      "summary": "Customer reports a duplicate charge on their credit card for a specific order.",
      "reasoning": "This is a clear billing issue involving a duplicate charge for a specific order, which is a high-priority financial concern requiring immediate attention."
    }}
    ```

    --- Current Ticket to Process ---

    Think step-by-step to ensure accuracy and adherence to the JSON format. Then provide the final JSON response.

    Ticket: """{ticket_text}"""
    """

    try:
        # Call the LLM client with the constructed prompt
        llm_response_str = client.generate(prompt, temperature=0.1) # Lower temperature for more deterministic output
        
        # Attempt to parse the JSON response
        parsed_response = json.loads(llm_response_str)
        return parsed_response
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from LLM: {e}")
        print(f"LLM Raw Response: {llm_response_str}")
        # Fallback for malformed JSON, or attempt to re-prompt
        return {
            "category": "General Inquiry",
            "sub_category": "Parsing Error",
            "priority": "High",
            "entities": [],
            "summary": "Failed to parse LLM response. Original ticket: {ticket_text[:50]}...",
            "reasoning": f"The LLM returned malformed JSON. Original error: {e}"
        }
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return {
            "category": "General Inquiry",
            "sub_category": "System Error",
            "priority": "High",
            "entities": [],
            "summary": "An unexpected system error occurred during ticket processing. Original ticket: {ticket_text[:50]}...",
            "reasoning": f"An unexpected error occurred: {e}"
        }

# --- Test your implementation with sample tickets ---
print("\n--- Processing Sample Tickets ---")
for i, ticket in enumerate(sample_tickets):
    print(f"\n--- Ticket {i+1} ---")
    print(f"Input: {ticket}")
    result = classify_ticket(ticket, llm_client)
    print("Output (JSON):")
    print(json.dumps(result, indent=2))
    
    # Basic validation (optional, but good practice)
    if not isinstance(result, dict):
        print("Validation Error: Output is not a dictionary.")
    elif "category" not in result or "priority" not in result or "entities" not in result or "summary" not in result or "reasoning" not in result:
        print("Validation Error: Missing required fields in JSON output.")
    else:
        print("Validation: Basic JSON structure looks good.")

print("\n--- Exercise Complete --- ")
print("Review the outputs and compare them against the requirements. Iterate on your prompt to improve accuracy and robustness!")
